# Clean Letterboxd Split CSVs

This notebook:
1. Reads `letterboxd_split_1.csv` through `letterboxd_split_22.csv`
2. Drops rows with null `num_ratings` or `avg_rating`
3. Drops rows with no `genre`
4. Combines everything into one big CSV
5. Resplits the combined data into 22 files inside a `cleaned_data/` folder

In [1]:
import os
import glob
import math
import pandas as pd
import numpy as np

In [2]:
# ---- Config ----
# Folder where the letterboxd_split_*.csv files live (defaults to the notebook's directory).
INPUT_DIR = "final_data/letterboxd_review_data"

# Number of input split files (1..N)
NUM_SPLITS = 22

# Output folder for the cleaned, re-split CSVs
OUTPUT_DIR = os.path.join(INPUT_DIR, "cleaned_data")

# Path for the single combined CSV (kept alongside the splits)
COMBINED_PATH = os.path.join(INPUT_DIR, "letterboxd_cleaned_combined.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Input dir : {os.path.abspath(INPUT_DIR)}")
print(f"Output dir: {os.path.abspath(OUTPUT_DIR)}")

Input dir : /Users/adonisgeoffmacias/Documents/GitHub/data-trio-project/final_data/letterboxd_review_data/final_data/letterboxd_review_data
Output dir: /Users/adonisgeoffmacias/Documents/GitHub/data-trio-project/final_data/letterboxd_review_data/final_data/letterboxd_review_data/cleaned_data


In [3]:
# ---- Step 1: Read + clean each split ----
def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    """Drop rows with null num_ratings/avg_rating, and rows with no genre."""
    # Drop rows where num_ratings or avg_rating are null
    df = df.dropna(subset=["num_ratings", "avg_rating"])

    # Drop rows with no genre. Treat empty strings and whitespace as missing.
    genre = df["genre"].astype("string").str.strip()
    df = df[genre.notna() & (genre != "") & (genre.str.lower() != "nan")]
    return df

cleaned_frames = []
stats = []

for i in range(1, NUM_SPLITS + 1):
    path = os.path.join(f"letterboxd_split_{i}.csv")
    if not os.path.exists(path):
        print(f"  [skip] {path} not found")
        continue

    df = pd.read_csv(path)

    # Cast tmdb_id and num_ratings to nullable integers so they write as 936075 instead of 936075.0
    for col in ["tmdb_id", "num_ratings"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    before = len(df)
    cleaned = clean_df(df)
    after = len(cleaned)
    stats.append((i, before, after, before - after))
    cleaned_frames.append(cleaned)
    print(f"  split {i:>2}: {before:>7} -> {after:>7}  (dropped {before-after})")

stats_df = pd.DataFrame(stats, columns=["split", "rows_before", "rows_after", "rows_dropped"])
stats_df.loc["total"] = ["-", stats_df["rows_before"].sum(), stats_df["rows_after"].sum(), stats_df["rows_dropped"].sum()]
stats_df

  split  1:  495746 ->  369713  (dropped 126033)
  split  2:  495746 ->  375805  (dropped 119941)
  split  3:  495746 ->  363162  (dropped 132584)
  split  4:  495746 ->  365077  (dropped 130669)
  split  5:  495746 ->  378134  (dropped 117612)
  split  6:  495746 ->  356214  (dropped 139532)
  split  7:  495746 ->  370819  (dropped 124927)
  split  8:  495746 ->  367732  (dropped 128014)
  split  9:  495746 ->  363390  (dropped 132356)
  split 10:  495746 ->  367792  (dropped 127954)
  split 11:  495746 ->  370206  (dropped 125540)
  split 12:  495746 ->  368269  (dropped 127477)
  split 13:  495746 ->  365839  (dropped 129907)
  split 14:  495746 ->  370719  (dropped 125027)
  split 15:  495746 ->  365314  (dropped 130432)
  split 16:  495746 ->  372019  (dropped 123727)
  split 17:  495746 ->  364789  (dropped 130957)
  split 18:  495746 ->  364363  (dropped 131383)
  split 19:  495746 ->  365556  (dropped 130190)
  split 20:  495746 ->  367241  (dropped 128505)
  split 21:  495746 

,split,rows_before,rows_after,rows_dropped
0,1,495746,369713,126033
1,2,495746,375805,119941
2,3,495746,363162,132584
3,4,495746,365077,130669
4,5,495746,378134,117612
5,6,495746,356214,139532
6,7,495746,370819,124927
7,8,495746,367732,128014
8,9,495746,363390,132356
9,10,495746,367792,127954


In [4]:
# ---- Step 2: Combine cleaned rows into one CSV ----
if not cleaned_frames:
    raise RuntimeError("No input files were read. Check INPUT_DIR and file names.")

combined = pd.concat(cleaned_frames, ignore_index=True)
combined.to_csv(COMBINED_PATH, index=False)

print(f"Combined shape: {combined.shape}")
print(f"Saved combined CSV -> {COMBINED_PATH}")
combined.head()

Combined shape: (7908280, 14)
Saved combined CSV -> final_data/letterboxd_review_data/letterboxd_cleaned_combined.csv


,username,tmdb_id,movie_title,rating,genre,subgenre_1,subgenre_2,subgenre_3,subgenre_4,subgenre_5,num_ratings,avg_rating,director_url,actors_url
0,schaffrillas,13352,It's a Very Merry Muppet Christmas Movie,2.5,Comedy,Family,NaN,NaN,NaN,NaN,15069,3.10,/director/kirk-r-thatcher/,"/actor/steve-whitmire/,/actor/dave-goelz/,/act..."
1,schaffrillas,13376,Eight Crazy Nights,1.5,Animation,Comedy,NaN,NaN,NaN,NaN,58041,2.38,/director/seth-kearsley/,"/actor/adam-sandler/,/actor/jackie-sandler/,/a..."
2,schaffrillas,413817,The Rapsittie Street Kids: Believe in Santa,0.5,Animation,Family,TV Movie,Music,NaN,NaN,7256,1.73,/director/colin-slater/,"/actor/jack-angel/,/actor/jodi-benson/,/actor/..."
3,schaffrillas,15909,Kermit's Swamp Years,1.0,Fantasy,Family,Comedy,NaN,NaN,NaN,4207,2.77,/director/david-gumpel/,"/actor/steve-whitmire/,/actor/bill-barretta/,/..."
4,schaffrillas,18357,The Country Bears,3.5,Adventure,Comedy,Family,NaN,NaN,NaN,16575,2.54,/director/peter-hastings/,"/actor/christopher-walken/,/actor/stephen-tobo..."


In [5]:
# ---- Step 3: Re-split combined data into NUM_SPLITS files inside cleaned_data/ ----
n_rows = len(combined)
chunk_size = math.ceil(n_rows / NUM_SPLITS)
print(f"Total rows: {n_rows} | chunk size: {chunk_size}")

written = []
for i in range(NUM_SPLITS):
    start = i * chunk_size
    end = min(start + chunk_size, n_rows)
    if start >= n_rows:
        # Still write an empty file with headers so we always have NUM_SPLITS outputs
        chunk = combined.iloc[0:0]
    else:
        chunk = combined.iloc[start:end]

    out_path = os.path.join(OUTPUT_DIR, f"letterboxd_split_{i+1}.csv")
    chunk.to_csv(out_path, index=False)
    written.append((i + 1, len(chunk), out_path))
    print(f"  wrote split {i+1:>2}: {len(chunk):>7} rows -> {out_path}")

pd.DataFrame(written, columns=["split", "rows", "path"])

Total rows: 7908280 | chunk size: 359468
  wrote split  1:  359468 rows -> final_data/letterboxd_review_data/cleaned_data/letterboxd_split_1.csv
  wrote split  2:  359468 rows -> final_data/letterboxd_review_data/cleaned_data/letterboxd_split_2.csv
  wrote split  3:  359468 rows -> final_data/letterboxd_review_data/cleaned_data/letterboxd_split_3.csv
  wrote split  4:  359468 rows -> final_data/letterboxd_review_data/cleaned_data/letterboxd_split_4.csv
  wrote split  5:  359468 rows -> final_data/letterboxd_review_data/cleaned_data/letterboxd_split_5.csv
  wrote split  6:  359468 rows -> final_data/letterboxd_review_data/cleaned_data/letterboxd_split_6.csv
  wrote split  7:  359468 rows -> final_data/letterboxd_review_data/cleaned_data/letterboxd_split_7.csv
  wrote split  8:  359468 rows -> final_data/letterboxd_review_data/cleaned_data/letterboxd_split_8.csv
  wrote split  9:  359468 rows -> final_data/letterboxd_review_data/cleaned_data/letterboxd_split_9.csv
  wrote split 10:  3594

,split,rows,path
0,1,359468,final_data/letterboxd_review_data/cleaned_data...
1,2,359468,final_data/letterboxd_review_data/cleaned_data...
2,3,359468,final_data/letterboxd_review_data/cleaned_data...
3,4,359468,final_data/letterboxd_review_data/cleaned_data...
4,5,359468,final_data/letterboxd_review_data/cleaned_data...
5,6,359468,final_data/letterboxd_review_data/cleaned_data...
6,7,359468,final_data/letterboxd_review_data/cleaned_data...
7,8,359468,final_data/letterboxd_review_data/cleaned_data...
8,9,359468,final_data/letterboxd_review_data/cleaned_data...
9,10,359468,final_data/letterboxd_review_data/cleaned_data...


In [6]:
# ---- Step 4: Sanity checks on the cleaned, re-split files ----
out_files = sorted(
    glob.glob(os.path.join(OUTPUT_DIR, "letterboxd_split_*.csv")),
    key=lambda p: int(os.path.basename(p).split("_")[-1].split(".")[0]),
)

total_rows = 0
bad = []
for p in out_files:
    d = pd.read_csv(p)
    total_rows += len(d)
    if d[["num_ratings", "avg_rating"]].isna().any().any():
        bad.append((p, "null num_ratings/avg_rating"))
    if len(d) and (d["genre"].astype("string").str.strip().fillna("") == "").any():
        bad.append((p, "empty genre"))

print(f"Files written: {len(out_files)}")
print(f"Total rows across cleaned splits: {total_rows}")
print(f"Combined CSV rows: {len(combined)}")
assert total_rows == len(combined), "Row count mismatch between combined and re-split files!"
if bad:
    print("PROBLEMS:", bad)
else:
    print("All cleaned splits pass the null/genre checks.")

Files written: 22
Total rows across cleaned splits: 7908280
Combined CSV rows: 7908280
All cleaned splits pass the null/genre checks.
